# LLM Clustering — Zero-Shot vs HAC

**Objective**: Compare how a local LLM (Llama 3.2 3B via Ollama) clusters financial news articles for a given period against our NLP pipelines (Word2Vec+HAC and FinBERT+HAC).

**Setup required** (one-time):
```bash
# 1. Install Ollama: https://ollama.com/download
# 2. Pull the model:
ollama pull llama3.2:3b
# 3. Start Ollama (if not already running):
ollama serve
```

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.abspath(os.path.join("..")))
from src.llm_clustering import (
    run_llm_clustering,
    compare_llm_vs_hac,
    print_comparison_summary,
)
from src.news_clustering import compute_cluster_centroids
from sklearn.preprocessing import normalize

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
import warnings

# Masquer les avertissements inutiles
warnings.filterwarnings("ignore")

# ===========================================
# 1. CHARGEMENT DU MODÈLE LLM
# ===========================================
print("Chargement de l'Analyste IA (Qwen 0.5B)...")
llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct", 
    device="cpu", 
    dtype=torch.float32, 
)

# ==========================================
# 2. CHARGEMENT DE LA BASE DE DONNÉES COMPLÈTE
# ==========================================
# On charge votre fichier global qui contient tous les articles (les 736)
# Remplacez le chemin par celui de votre fichier contenant au moins 'date' et 'headline'
df_global = pd.read_csv("../data/processed/news_2026_bpe.csv") 

# S'assurer que la colonne date est bien au format "Date" pour faciliter le filtrage
df_global['date'] = pd.to_datetime(df_global['date'])

# ==========================================
# 3. LE MOTEUR D'ANALYSE TEMPORELLE
# ==========================================
def analyser_periode_llm(start_date, end_date, question, max_articles=40):
    """
    Filtre les articles sur une période donnée et demande à l'IA de les analyser/clusteriser.
    """
    # 1. Filtrage temporel strict
    mask = (df_global['date'] >= pd.to_datetime(start_date)) & (df_global['date'] <= pd.to_datetime(end_date))
    df_periode = df_global.loc[mask].copy()
    
    if len(df_periode) == 0:
        return f"Aucun article trouvé entre le {start_date} et le {end_date}."
        
    # 2. Sécurité CPU / Contexte
    # Si la période est trop large, on limite pour ne pas faire exploser le CPU
    if len(df_periode) > max_articles:
        print(f"⚠️ Attention : {len(df_periode)} articles trouvés. Limitation aux {max_articles} premiers pour le CPU.")
        df_periode = df_periode.head(max_articles)
        
    # 3. Formatage du Contexte pour l'IA
    # On donne un numéro [ID] à chaque article pour que l'IA puisse s'y référer facilement
    context_lines = []
    for idx, row in enumerate(df_periode.itertuples(), 1):
        # row.date.strftime('%Y-%m-%d') permet d'afficher la date proprement
        context_lines.append(f"[{idx}] ({row.date.strftime('%Y-%m-%d')}) : {row.headline}")
        
    context_str = "\n".join(context_lines)
    
    # 4. Prompting ciblé pour le Clustering Zéro-Shot
    system_prompt = (
        "Tu es un Lead Analyste Financier expert. Ton rôle est de lire une liste de titres d'actualité, "
        "d'identifier les thèmes récurrents, de les regrouper logiquement (clustering), et de répondre "
        "à la question de l'utilisateur avec une structure claire."
    )
    
    user_prompt = (
        f"Voici les actualités financières de la période allant du {start_date} au {end_date} :\n"
        f"{context_str}\n\n"
        f"Question de l'utilisateur : {question}"
    )
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    
    prompt_formate = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 5. Génération
    print(f"--> L'IA analyse {len(df_periode)} articles (cela prendra quelques secondes/minutes)...")
    outputs = llm(
        prompt_formate,
        max_new_tokens=450, # On permet une réponse plus longue car l'IA doit lister des clusters
        temperature=0.2,    # Température basse : on veut de la logique rigoureuse, pas d'hallucination
        do_sample=True,
        pad_token_id=llm.tokenizer.eos_token_id
    )
    
    reponse = outputs[0]["generated_text"].split("<|im_start|>assistant\n")[-1].strip()
    return reponse

In [ ]:
# --- Définition de votre période ---
debut = "2026-02-25"
fin = "2026-03-01"

print(f"=== ANALYSE DE LA SEMAINE DU {debut} AU {fin} ===")

# Question 1 : Le Clustering Zéro-Shot
question_clustering = """
En combien de clusters principaux classerais-tu ces informations ? 
Pour chaque cluster, donne-moi :
1. Un titre court pour le sujet général.
2. Un résumé d'une phrase.
3. Les numéros [ID] des articles qui appartiennent à ce cluster.
"""

reponse_1 = analyser_periode_llm(debut, fin, question_clustering)
print("\n🤖 RÉPONSE IA (CLUSTERING) :\n")
print(reponse_1)

print("-" * 50)

# Question 2 : Le Sentiment global de la période
question_sentiment = """
Au regard de l'ensemble de ces informations, comment qualifierais-tu la dynamique globale des marchés sur cette période (Neutre, Positif, Négatif) ?
"""

reponse_2 = analyser_periode_llm(debut, fin, question_sentiment)
print("\n🤖 RÉPONSE IA (SENTIMENT MACRO) :\n")
print(reponse_2)